In [3]:
import torch
import pandas as pd
import numpy as np
from datasets import load_dataset
from pyvene import (
    IntervenableModel,
    VanillaIntervention,
    RepresentationConfig,
    IntervenableConfig,
    ConstantSourceIntervention,
    LocalistRepresentationIntervention,
    create_gpt2,
    embed_to_distrib
)
from tqdm import tqdm
from plotnine import *
import matplotlib.pyplot as plt
import random

from pprint import pprint
import os

In [4]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"

print("Using device:", device)

model_names = {
    "base": "gpt2-xl",
    "fine_tuned": "utahnlp/boolq_gpt2-xl_seed-1"
}

Using device: cuda:0


In [5]:
class NoiseIntervention(ConstantSourceIntervention, LocalistRepresentationIntervention):
    def __init__(self, embed_dim, **kwargs):
        super().__init__()
        self.interchange_dim = embed_dim
        self.noise_level = 0.13462981581687927
        self.noise = None

    def forward(self, base, source=None, subspaces=None):
        current_len = base.shape[1]

        if self.noise is None or self.noise.shape[1] != current_len:
            rs = np.random.RandomState(1)
            prng = lambda *shape: rs.randn(*shape)
            self.noise = torch.from_numpy(
                prng(1, current_len, self.interchange_dim)
            ).to(base.device)
            
        base[..., : self.interchange_dim] += self.noise * self.noise_level
        return base

# Funkcje pomocnicze

In [6]:
def get_range_of_subject(tokenizer, subject, full_prompt):
    # znajdujemy indeksy tokenów podmiotu
    full_ids = tokenizer.encode(full_prompt)
    subject_ids = tokenizer.encode(subject)
    
    # Szukamy podciągu subject_ids w full_ids
    len_sub = len(subject_ids)
    for i in range(len(full_ids) - len_sub + 1):
        if full_ids[i : i + len_sub] == subject_ids:
            return list(range(i, i + len_sub))
    # Zakładamy, że podmiot jest na początku zdania dla tych przykładów
    return list(range(len(subject_ids)))

# Funkcja pomocnicza do generowania configu
def restore_corrupted_with_interval_config(layer, stream="mlp_activation", window=10, num_layers=48):
    start = max(0, layer - window // 2)
    end = min(num_layers, layer - (-window // 2))
    config = IntervenableConfig(
        representations=[
            RepresentationConfig(0, "block_input"), # Warstwa do zaszumiania
        ] + [
            RepresentationConfig(i, stream) for i in range(start, end) # Warstwy do przywracania
        ],
        intervention_types=[NoiseIntervention] + [VanillaIntervention] * (end - start),
    )
    return config

In [7]:
dataset = load_dataset("google/boolq", split='train')
examples = dataset.shuffle(seed=42).select(range(10))

In [8]:
def print_dataset(examples):
    print(examples)
    for ex in examples:
        pprint(ex)

print_dataset(examples)

Dataset({
    features: ['question', 'answer', 'passage'],
    num_rows: 10
})
{'answer': False,
 'passage': "Henry Daniel Mills is a fictional character in ABC's television "
            'series Once Upon a Time. Henry is the boy Emma Swan gave up to '
            'adoption; Regina Mills adopted him. Henry was originally '
            'portrayed as a child by Jared S. Gilmore, who won the Young '
            'Artist Award for Best Performance in a TV Series -- Leading Young '
            "Actor in 2012. For the show's seventh and final season, Andrew J. "
            'West later took over the role of Henry as an adult and father to '
            'a eight-year-old girl named Lucy, with Gilmore also making three '
            'appearances as Henry during the season.',
 'question': 'did henry die in once upon a time'}
{'answer': False,
 'passage': 'Straight Talk offers a variety of prepaid, no contract, phones on '
            'their website for use with their plans. Straight Talk also a

In [9]:
examples_no_passage = examples.remove_columns("passage")
print_dataset(examples_no_passage)

Dataset({
    features: ['question', 'answer'],
    num_rows: 10
})
{'answer': False, 'question': 'did henry die in once upon a time'}
{'answer': False, 'question': 'can i use a tracfone with straight talk service'}
{'answer': False,
 'question': 'has any nba team ever come back from 3-0 in playoffs'}
{'answer': False, 'question': 'netball can you shoot from outside the circle'}
{'answer': True,
 'question': 'is the schwarzschild radius the same as the event horizon'}
{'answer': True, 'question': 'is shark tank a copy of dragons den'}
{'answer': False,
 'question': 'is there a difference between kava and kava kava'}
{'answer': True, 'question': 'has a mlb game ever ended in a tie'}
{'answer': True, 'question': 'can there be a hurricane in the pacific ocean'}
{'answer': True, 'question': 'is season 5 the last season of young and hungry'}


In [10]:
subjects = [
    'henry',
    'tracfone',
    'nba team',
    'netball',
    'schwarzschild radius',
    'shark tank',
    'kava',
    'mlb game',
    'hurricane',
    'season 5',
]
examples = examples.add_column("subject", subjects)
examples_no_passage = examples_no_passage.add_column("subject", subjects)

In [11]:
print_dataset(examples_no_passage)

Dataset({
    features: ['question', 'answer', 'subject'],
    num_rows: 10
})
{'answer': False,
 'question': 'did henry die in once upon a time',
 'subject': 'henry'}
{'answer': False,
 'question': 'can i use a tracfone with straight talk service',
 'subject': 'tracfone'}
{'answer': False,
 'question': 'has any nba team ever come back from 3-0 in playoffs',
 'subject': 'nba team'}
{'answer': False,
 'question': 'netball can you shoot from outside the circle',
 'subject': 'netball'}
{'answer': True,
 'question': 'is the schwarzschild radius the same as the event horizon',
 'subject': 'schwarzschild radius'}
{'answer': True,
 'question': 'is shark tank a copy of dragons den',
 'subject': 'shark tank'}
{'answer': False,
 'question': 'is there a difference between kava and kava kava',
 'subject': 'kava'}
{'answer': True,
 'question': 'has a mlb game ever ended in a tie',
 'subject': 'mlb game'}
{'answer': True,
 'question': 'can there be a hurricane in the pacific ocean',
 'subject': 'hur

In [12]:
titles = {
    "block_output": "Single restored layer",
    "mlp_activation": "Center of interval of 10 patched MLP layers",
    "attention_output": "Center of interval of 10 patched Attn layers"
}
colors = {"block_output": "Purples", "mlp_activation": "Greens", "attention_output": "Reds"}


def process_model(model_name, model_acronym, examples, streams_to_plot, save_dir="causal_tracing"):
    def get_filename(idx, stream):
        return f"{save_dir}/{model_acronym}_{idx}_{stream}.pdf"
    
    def file_exists(idx, stream):
        filename = get_filename(idx, stream)
        return os.path.exists(filename)
    
    def files_exist(idx):
        for stream in streams_to_plot:
            if not file_exists(idx, stream):
                return False
        return True
    
    config, tokenizer, gpt = create_gpt2(name=model_name)
    gpt.to(device)
    gpt.eval()
    
    for i, ex in enumerate(examples):
        if files_exist(i):
            print(f"Files for example {i} already exist. Skipping...")
            continue
        
        full_prompt = f'{ex["question"]}? Answer (Yes/No): '
        answer = "Yes" if ex["answer"] else "No"
        
        print(f"\n--- Processing Example {i+1}: '{full_prompt}' -> '{answer}' ---")
        
        base_tensors = tokenizer(full_prompt, return_tensors="pt").to(device)
        target_id = tokenizer.encode(answer)[0]
        subject_indices = get_range_of_subject(tokenizer, ex["subject"], full_prompt)
        print(f"Subject tokens indices: {subject_indices}")
        
        for stream in streams_to_plot:
            if file_exists(i, stream):
                continue
            data = []
            n_layers = gpt.config.n_layer
            n_tokens = base_tensors.input_ids.shape[1]
            
            for layer_i in tqdm(range(n_layers), desc=f"Scanning {stream}"):
                for pos_i in range(n_tokens):
                    config = restore_corrupted_with_interval_config(
                        layer_i, stream, 
                        window=1 if stream == "block_output" else 10
                    )
                    n_restores = len(config.representations) - 1
                    
                    intervenable = IntervenableModel(config, gpt)
                    
                    _, counterfactual_outputs = intervenable(
                        base_tensors,
                        [None] + [base_tensors] * n_restores,
                        {
                            "sources->base": (
                                [None] + [[[pos_i]]] * n_restores,
                                # POPRAWIONE ZAGNIEŻDŻENIE:
                                [[subject_indices]] + [[[pos_i]]] * n_restores, 
                            )
                        },
                    )
                    
                    distrib = embed_to_distrib(gpt, counterfactual_outputs.last_hidden_state, logits=False)
                    prob = distrib[0][-1][target_id].detach().cpu().item()
                    data.append({"layer": layer_i, "pos": pos_i, "prob": prob})
            
            df = pd.DataFrame(data)
            token_labels = [tokenizer.decode([x]) for x in base_tensors.input_ids[0]]
            token_labels = [f"{lbl} ({j})" for j, lbl in enumerate(token_labels)]
            
            plot = (
                ggplot(df, aes(x="layer", y="pos"))
                + geom_tile(aes(fill="prob"))
                + scale_fill_cmap(colors[stream])
                + xlab(f"{titles[stream]} \n Example: {ex['subject']}...")
                + scale_y_reverse(
                    limits=(-0.5, n_tokens - 0.5),
                    breaks=range(n_tokens),
                    labels=token_labels
                )
                + theme(figure_size=(6, 4))
                + ylab("Token Position")
                + theme(axis_text_y=element_text(angle=0, hjust=1))
                + ggtitle(f"Target: {answer}")
            )
            print(plot)
            ggsave(plot, filename=get_filename(i, stream), dpi=200)
    del gpt

In [ ]:
streams_to_plot = [
    "mlp_activation",
    "block_output",
    "attention_output",
]

for model_acronym, model_name in model_names.items():
    print(f"Loading model: {model_name}")
    process_model(model_name, model_acronym, examples, streams_to_plot)

Loading model: gpt2-xl
loaded model
Files for example 0 already exist. Skipping...
Loading model: utahnlp/boolq_gpt2-xl_seed-1


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

loaded model

--- Processing Example 1: 'did henry die in once upon a time? Answer (Yes/No): ' -> 'No' ---
Subject tokens indices: [0, 1]


Scanning mlp_activation:   0%|          | 0/48 [00:15<?, ?it/s]


KeyboardInterrupt: 

In [24]:
print(examples[0])
print(examples[:1])

_examples = [examples[:1]]

print(_examples)

for i, ex in enumerate(_examples):
     
    full_prompt = f'{ex["question"]}? Answer (Yes/No): '
    answer = "Yes" if ex["answer"] else "No"
    
    print(ex["question"])
    print(ex["subject"])
    print(ex["answer"])
    print("Full prompt:", full_prompt)
    print("Expected answer:", answer)

{'question': 'did henry die in once upon a time', 'answer': False, 'passage': "Henry Daniel Mills is a fictional character in ABC's television series Once Upon a Time. Henry is the boy Emma Swan gave up to adoption; Regina Mills adopted him. Henry was originally portrayed as a child by Jared S. Gilmore, who won the Young Artist Award for Best Performance in a TV Series -- Leading Young Actor in 2012. For the show's seventh and final season, Andrew J. West later took over the role of Henry as an adult and father to a eight-year-old girl named Lucy, with Gilmore also making three appearances as Henry during the season.", 'subject': 'henry'}
{'question': ['did henry die in once upon a time'], 'answer': [False], 'passage': ["Henry Daniel Mills is a fictional character in ABC's television series Once Upon a Time. Henry is the boy Emma Swan gave up to adoption; Regina Mills adopted him. Henry was originally portrayed as a child by Jared S. Gilmore, who won the Young Artist Award for Best Per